## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-23-22-52-12 +0000,wapo,Van Hollen uses Huang controversy in bid to ou...,https://www.washingtonpost.com/dc-md-va/2026/1...
1,2026-11-23-22-49-00 +0000,wsj,Opinion | Sanctioning People for Building Home...,https://www.wsj.com/opinion/sanctioning-people...
2,2026-11-23-22-45-51 +0000,bbc,US to blacklist Asian AI firm Megaspeed after ...,https://www.bbc.co.uk/news/articles/cv98xpg5dg...
3,2026-11-23-22-45-03 +0000,nypost,Eric Ripert’s Le Bernardin named best of the b...,https://nypost.com/2026/11/23/business/eric-ri...
4,2026-11-23-22-42-49 +0000,wapo,A shocked Washington preps expanded investigat...,https://www.washingtonpost.com/politics/2026/1...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2610/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
3,huang,50
18,nvidia,43
143,trump,38
159,jensen,37
146,ceo,20


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
190,2026-11-23-16-12-26 +0000,nypost,Trump assured Nvidia board Jensen Huang unlike...,https://nypost.com/2026/11/23/us-news/trump-as...,208
113,2026-11-23-19-46-00 +0000,wsj,When President Trump wanted Jensen Huang back ...,https://www.wsj.com/politics/policy/nvidia-ceo...,203
193,2026-11-23-15-53-43 +0000,nypost,Trump suggests Jensen Huang could have moved t...,https://nypost.com/2026/11/23/us-news/trump-su...,201
175,2026-11-23-17-17-17 +0000,nypost,Trump posts edited image of assassination atte...,https://nypost.com/2026/11/23/us-news/trump-po...,198
155,2026-11-23-18-08-18 +0000,nyt,Trump Draws Bipartisan Criticism for Pushing t...,https://www.nytimes.com/2026/11/23/us/politics...,198


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
190,208,2026-11-23-16-12-26 +0000,nypost,Trump assured Nvidia board Jensen Huang unlike...,https://nypost.com/2026/11/23/us-news/trump-as...
298,69,2026-11-23-09-47-02 +0000,cbc,Zelenskyy confirms Ukraine was behind attack o...,https://www.cbc.ca/news/world/ukraine-attack-r...
198,65,2026-11-23-15-42-21 +0000,nypost,Nvidia adds nearly $1 trillion in market value...,https://nypost.com/2026/11/23/business/nvidia-...
16,65,2026-11-23-22-29-00 +0000,wsj,"Rep. Jimmy Gomez (D., Calif.) said he would re...",https://www.wsj.com/politics/democratic-lawmak...
294,54,2026-11-23-10-00-00 +0000,wsj,Democrats making presidential bids will gather...,https://www.wsj.com/politics/elections/democra...
131,47,2026-11-23-19-12-00 +0000,wsj,The U.S. Army said it plans to detonate severa...,https://www.wsj.com/politics/national-security...
264,38,2026-11-23-11-54-00 +0000,wsj,Texas Pacific Land to Buy EagleRock Land in $3...,https://www.wsj.com/business/energy-oil/texas-...
64,37,2026-11-23-21-26-16 +0000,nypost,California ICE arrest caught on shocking video...,https://nypost.com/2026/11/23/us-news/video-sh...
84,37,2026-11-23-21-01-51 +0000,nypost,"Virginia, Oregon sue Elon Musk’s SpaceX over G...",https://nypost.com/2026/11/23/business/virgini...
93,34,2026-11-23-20-30-16 +0000,nypost,Russia has lost astonishing 700K soldiers sinc...,https://nypost.com/2026/11/23/world-news/russi...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
